# Experiments Section 9.2 — Table 3: IHDP semi-synthetic ATE benchmark

This notebook reproduces the IHDP semi-synthetic ATE benchmark in Section 9.2 and Appendix R. It uses the observational training split with 100 replications, two-fold cross-fitting, and the methods reported in Table 3: Time-SMR, Joint-SMR, SQ-Riesz, BKL-Riesz, and logistic MLE/AIPW.

In [ ]:

from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd()
for parent in [REPO, *REPO.parents]:
    if (parent / "src" / "genriesz" / "scorematchingriesz.py").exists():
        REPO = parent
        break
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import genriesz.scorematchingriesz as smr
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
print("device:", DEVICE)


In [ ]:
from sklearn.linear_model import LogisticRegression

IHDP_FILE = REPO / "notebooks" / "scorematchingriesz" / "data" / "ihdp" / "ihdp_npci_1-100.train.npz"
N_REPLICATIONS = 100
N_FOLDS = 2
HIDDEN_DIMS = (256, 256, 256)
OUTCOME_EPOCHS = 150
RATIO_STEPS = 2000
BATCH_SIZE = 256
INTEGRATION_STEPS = 128
CLIP_LOG_RATIO = 20.0
TABLE_TITLE = "Table 3: IHDP semi-synthetic ATE benchmark"
FIGURE_TITLE = "IHDP ATE estimation errors"

In [ ]:

def load_ihdp_replication(path, rep):
    raw = np.load(path)
    idx = int(rep)
    z = raw["x"][:, :, idx].astype("float32")
    d = raw["t"][:, idx].astype("float32").reshape(-1)
    y = raw["yf"][:, idx].astype("float32").reshape(-1)
    mu0 = raw["mu0"][:, idx].reshape(-1)
    mu1 = raw["mu1"][:, idx].reshape(-1)
    theta = float(np.mean(mu1 - mu0))
    x = np.column_stack([d, z]).astype("float32")
    return {"X": x, "Z": z, "D": d, "Y": y, "theta": theta}


def switch_d(x, value):
    out = np.asarray(x, dtype="float32").copy()
    out[:, 0] = float(value)
    return out


def summarize_trials(df, group_cols):
    return df.groupby(group_cols).agg(
        trials=("estimate", "count"), truth_mean=("truth", "mean"), bias=("error", "mean"),
        mae=("error", lambda s: np.mean(np.abs(s))), rmse=("error", lambda s: np.sqrt(np.mean(np.square(s)))),
        coverage=("covered", "mean"), avg_se=("se", "mean")
    ).reset_index()


In [ ]:

METHODS = ["Time-SMR", "Joint-SMR", "SQ-Riesz", "BKL-Riesz", "MLE"]


def ratio_z(method, z_all, z_group, z_eval, seed):
    if method == "Time-SMR":
        model = smr.fit_time_smr_dre_infinity(z_all, z_group, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
        log_r = smr.log_ratio_from_time_score(model, z_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=z_group, device=DEVICE)
        return np.exp(np.clip(log_r.detach().cpu().numpy().reshape(-1), -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
    if method == "Joint-SMR":
        model = smr.fit_joint_smr_dre_infinity(z_all, z_group, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
        log_r = smr.log_ratio_from_joint_time_head(model, z_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=z_group, device=DEVICE)
        return np.exp(np.clip(log_r.detach().cpu().numpy().reshape(-1), -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
    if method == "SQ-Riesz":
        model = smr.fit_sq_riesz_ratio(z_all, z_group, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
        return smr.eval_ratio_sq(model, z_eval, normalize=True, x_p_for_norm=z_group, device=DEVICE).reshape(-1)
    if method == "BKL-Riesz":
        model = smr.fit_bkl_riesz_ratio(z_all, z_group, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
        return smr.eval_ratio_bkl(model, z_eval, normalize=True, x_p_for_norm=z_group, device=DEVICE).reshape(-1)
    raise ValueError(method)


def estimate_ihdp_replication(rep):
    data = load_ihdp_replication(IHDP_FILE, rep)
    x, z, d, y, theta = data["X"], data["Z"], data["D"], data["Y"], data["theta"]
    rows = []
    for method in METHODS:
        score_values = np.zeros(len(y))
        for train_idx, test_idx in smr.crossfit_splits(len(y), n_folds=N_FOLDS, seed=RANDOM_SEED + rep):
            x_train, z_train, d_train, y_train = x[train_idx], z[train_idx], d[train_idx], y[train_idx]
            x_test, z_test, d_test, y_test = x[test_idx], z[test_idx], d[test_idx], y[test_idx]
            outcome = smr.fit_outcome_net(x_train, y_train, hidden_dims=HIDDEN_DIMS, n_epochs=OUTCOME_EPOCHS, batch_size=BATCH_SIZE, seed=RANDOM_SEED + rep, device=DEVICE)
            gamma_hat = smr.predict_outcome(outcome, x_test, device=DEVICE).reshape(-1)
            m_gamma = smr.predict_outcome(outcome, switch_d(x_test, 1), device=DEVICE).reshape(-1) - smr.predict_outcome(outcome, switch_d(x_test, 0), device=DEVICE).reshape(-1)
            pi1 = float(np.mean(d_train))
            if method == "MLE":
                clf = LogisticRegression(max_iter=2000, solver="lbfgs")
                clf.fit(z_train, d_train.astype(int))
                e_hat = np.clip(clf.predict_proba(z_test)[:, 1], 1e-3, 1 - 1e-3)
                alpha_hat = d_test / e_hat - (1.0 - d_test) / (1.0 - e_hat)
            else:
                z_treated = z_train[d_train > 0.5]
                z_control = z_train[d_train <= 0.5]
                r1 = ratio_z(method, z_train, z_treated, z_test, RANDOM_SEED + rep)
                r0 = ratio_z(method, z_train, z_control, z_test, RANDOM_SEED + 1000 + rep)
                alpha_hat = d_test * r1 / pi1 - (1.0 - d_test) * r0 / (1.0 - pi1)
            score_values[test_idx] = m_gamma + alpha_hat * (y_test - gamma_hat)
        est = smr.wald_interval(score_values)
        rows.append({"method": method, "estimate": est.estimate, "se": est.se, "ci_low": est.ci_low, "ci_high": est.ci_high, "truth": theta, "error": est.estimate - theta, "covered": est.ci_low <= theta <= est.ci_high})
    return rows

all_rows = []
for rep in range(N_REPLICATIONS):
    all_rows.extend([{**row, "replication": rep} for row in estimate_ihdp_replication(rep)])
ihdp_results = pd.DataFrame(all_rows)
print(TABLE_TITLE)
display(summarize_trials(ihdp_results, ["method"]))


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
methods = list(ihdp_results["method"].unique())
ax.boxplot([ihdp_results.loc[ihdp_results["method"] == m, "error"] for m in methods], labels=methods, showfliers=False)
ax.axhline(0.0, linestyle="--")
ax.set_title(FIGURE_TITLE)
ax.set_ylabel("estimate minus truth")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()
